# ZooGotchi S3 — On-Chip Safe PLE Micro-LLM Trainer (Google Colab T4)

Trains an ultra-compact **Per-Layer Embedding (PLE) + Asymmetric Vocabulary** language model (~9M params quantized to 4-bit `zoo_pet.bin`) that runs **100% standalone on your Waveshare ESP32-S3-Touch-LCD-1.69** (no PC or cloud needed after flashing!).

### Safety Guarantees Baked Into Weights:
1. Trained strictly on wholesome zoo-animal & Karimono conversations.
2. Asymmetric output head (`V_out = 1421` vetted tokens from `include/zoo_vocab_out.h`) — unsafe words have **no output token ID**.

In [ ]:
#@title 1. Setup & Upload/Verify Dataset + Vocab
!pip install -q torch numpy
import json, math, os, re, struct, zlib
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

# Upload vocab_out.json and zoo_seed_conversations.jsonl (or batch output) from llm_training/
V_IN = 2048
D_MODEL = 192
N_LAYERS = 6
N_HEADS = 4
N_KV_HEADS = 4
HIDDEN_DIM = 512
MAX_SEQ_LEN = 256
GROUP_SIZE = 128

with open('vocab_out.json', 'r', encoding='utf-8') as f:
    vout_words = json.load(f)
V_OUT = len(vout_words)
word2out = {w.lower(): i for i, w in enumerate(vout_words)}
print(f'Loaded V_OUT={V_OUT} safe output tokens')

def hash_word_to_vin(word: str, v_in: int = V_IN) -> int:
    """Exact FNV-1a hash matching src/ai/micro_llm_ple.cpp"""
    h = 2166136261
    for ch in word.lower().encode('ascii', 'ignore'):
        h ^= ch
        h = (h * 16777619) & 0xFFFFFFFF
    return 4 + (h % (v_in - 4))


In [ ]:
#@title 2. Define PLE + Asymmetric Output Transformer (Matches micro_llm_ple.cpp)
class RMSNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return self.weight * x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-5)

def apply_rope(q, k):
    B, T, nh, hd = q.shape
    pos = torch.arange(T, device=q.device).unsqueeze(1)
    dim_idx = torch.arange(0, hd, 2, device=q.device).float()
    freq = 1.0 / (10000.0 ** (dim_idx / hd))
    ang = pos * freq
    cos = torch.cos(ang)[None, :, None, :]
    sin = torch.sin(ang)[None, :, None, :]
    def rot(t):
        t0, t1 = t[..., 0::2], t[..., 1::2]
        return torch.stack([t0 * cos - t1 * sin, t0 * sin + t1 * cos], dim=-1).flatten(-2)
    return rot(q), rot(k)

class PleBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.att_norm = RMSNorm(D_MODEL)
        self.wq = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.wk = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.wv = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.wo = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.ffn_norm = RMSNorm(D_MODEL)
        self.w1 = nn.Linear(D_MODEL, HIDDEN_DIM, bias=False)
        self.w3 = nn.Linear(D_MODEL, HIDDEN_DIM, bias=False)
        self.w2 = nn.Linear(HIDDEN_DIM, D_MODEL, bias=False)
    def forward(self, x, ple_l):
        x = x + ple_l
        B, T, D = x.shape
        hd = D // N_HEADS
        h = self.att_norm(x)
        q = self.wq(h).view(B, T, N_HEADS, hd)
        k = self.wk(h).view(B, T, N_HEADS, hd)
        v = self.wv(h).view(B, T, N_HEADS, hd)
        q, k = apply_rope(q, k)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(hd)
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        att = F.softmax(att.masked_fill(mask, float('-inf')), dim=-1)
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, D)
        x = x + self.wo(y)
        h = self.ffn_norm(x)
        x = x + self.w2(F.silu(self.w1(h)) * self.w3(h))
        return x

class ZooPleLLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.ple = nn.Embedding(V_IN, N_LAYERS * D_MODEL)
        self.layers = nn.ModuleList([PleBlock() for _ in range(N_LAYERS)])
        self.out_norm = RMSNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, V_OUT, bias=False)
    def forward(self, x_ids):
        B, T = x_ids.shape
        ple_all = self.ple(x_ids).view(B, T, N_LAYERS, D_MODEL)
        x = torch.zeros(B, T, D_MODEL, device=x_ids.device)
        for l, blk in enumerate(self.layers):
            x = blk(x, ple_all[:, :, l, :])
        return self.head(self.out_norm(x))

model = ZooPleLLM().to(device)
print('Total params:', sum(p.numel() for p in model.parameters()))

In [ ]:
#@title 3. Train on Wholesome Zoo Conversations
samples = []
with open('zoo_seed_conversations.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        p_ids = [1] + [hash_word_to_vin(w) for w in row['prompt'].split()]
        r_toks = re.findall(r"\.\.\.|[A-Za-z0-9'-]+|[.,!?:;()]", row['reply'])
        r_out = [word2out.get(t.lower(), 3) for t in r_toks] + [2]  # <eos>=2
        r_in  = [1] + [hash_word_to_vin(t) for t in r_toks]
        seq_in = (p_ids + r_in)[:MAX_SEQ_LEN]
        targets = ([-100] * len(p_ids) + r_out)[:len(seq_in)]
        if len(targets) < len(seq_in):
            targets += [-100] * (len(seq_in) - len(targets))
        samples.append((seq_in, targets))

opt = torch.optim.AdamW(model.parameters(), lr=6e-4, weight_decay=0.01)
model.train()
for epoch in range(25):
    tot_loss = 0.0
    for seq_in, tgt in samples:
        x = torch.tensor([seq_in], dtype=torch.long, device=device)
        y = torch.tensor([tgt], dtype=torch.long, device=device)
        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, V_OUT), y.view(-1), ignore_index=-100)
        opt.zero_grad(); loss.backward(); opt.step()
        tot_loss += loss.item()
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1}: avg loss = {tot_loss/max(1,len(samples)):.4f}')

In [ ]:
#@title 4. Quantize INT4 Group-128 & Export zoo_pet.bin (ZPLE v1) + Parity Check
def quantize_q4_matrix(w: np.ndarray, gs: int = 128) -> tuple[bytes, np.ndarray]:
    """Symmetric INT4 [-8..+7] stored +8 in [0..15] + IEEE754 fp16 scale per group."""
    rows, cols = w.shape
    out = bytearray()
    deq = np.zeros_like(w, dtype=np.float32)
    for r in range(rows):
        for g_start in range(0, cols, gs):
            grp = np.zeros(gs, dtype=np.float32)
            end = min(cols, g_start + gs)
            grp[:end - g_start] = w[r, g_start:end]
            amax = float(np.max(np.abs(grp)))
            scale = np.float16(amax / 7.0 if amax > 1e-8 else 1e-8)
            sc_f = float(scale)
            q = np.clip(np.round(grp / sc_f), -8, 7).astype(np.int8)
            deq[r, g_start:end] = q[:end - g_start] * sc_f
            u = (q + 8).astype(np.uint8)
            packed = (u[0::2] & 0x0F) | ((u[1::2] & 0x0F) << 4)
            out.extend(packed.tobytes())
            out.extend(struct.pack('<H', scale.view(np.uint16)))
    return bytes(out), deq

model.eval()
sd = {k: v.detach().cpu().numpy().astype(np.float32) for k, v in model.state_dict().items()}
ple_bytes = bytearray()
ple_w = sd['ple.weight'].reshape(V_IN, N_LAYERS, D_MODEL)
for t in range(V_IN):
    for l in range(N_LAYERS):
        b, _ = quantize_q4_matrix(ple_w[t, l:l+1, :], GROUP_SIZE)
        ple_bytes.extend(b)

dense_bytes = bytearray()
for l in range(N_LAYERS):
    dense_bytes.extend(sd[f'layers.{l}.att_norm.weight'].tobytes())
    for k in ('wq', 'wk', 'wv', 'wo'):
        b, _ = quantize_q4_matrix(sd[f'layers.{l}.{k}.weight'], GROUP_SIZE)
        dense_bytes.extend(b)
    dense_bytes.extend(sd[f'layers.{l}.ffn_norm.weight'].tobytes())
    for k in ('w1', 'w3', 'w2'):
        b, _ = quantize_q4_matrix(sd[f'layers.{l}.{k}.weight'], GROUP_SIZE)
        dense_bytes.extend(b)
dense_bytes.extend(sd['out_norm.weight'].tobytes())

head_bytes, head_deq = quantize_q4_matrix(sd['head.weight'], GROUP_SIZE)
head_rmse = float(np.sqrt(np.mean((sd['head.weight'] - head_deq) ** 2)))
print(f'Output head INT4 quantization RMSE: {head_rmse:.6f}')

vocab_offset = 64
ple_offset = vocab_offset
dense_offset = ple_offset + len(ple_bytes)
head_offset = dense_offset + len(dense_bytes)
total_bytes = head_offset + len(head_bytes)
payload = bytes(ple_bytes) + bytes(dense_bytes) + head_bytes
crc = zlib.crc32(payload) & 0xFFFFFFFF

hdr = struct.pack('<4sI10H6I12s', b'ZPLE', 1,
    D_MODEL, N_LAYERS, N_HEADS, N_KV_HEADS, HIDDEN_DIM, MAX_SEQ_LEN,
    V_IN, V_OUT, GROUP_SIZE, D_MODEL,
    vocab_offset, ple_offset, dense_offset, head_offset, total_bytes, crc, b'\x00'*12)
assert len(hdr) == 64
with open('zoo_pet.bin', 'wb') as f:
    f.write(hdr)
    f.write(payload)
print(f'Exported zoo_pet.bin ({total_bytes / 1024 / 1024:.2f} MB) <= 9.87 MB llm_model partition')
print('Flash onto Waveshare ESP32-S3 at offset 0x610000:')
print('  esptool.py --chip esp32s3 write_flash 0x610000 zoo_pet.bin')